# Notebook 4.4: Reducing Data with Principal Component Analysis

**Companion to Chapter 4: Implementing Data Pre-processing in Python**  
*Machine Learning with Python: Principles and Practical Techniques*

> **Estimated time:** 45–55 minutes  
> **Level:** Beginner to intermediate  
> **Environment:** Google Colab or Jupyter Notebook

---

## Related chapter ideas

This notebook develops feature extraction and dimensionality reduction using Principal Component Analysis (PCA). It continues with the student-success dataset used throughout the Chapter 4 notebook sequence.

## Learning objectives

By the end of this notebook, you will be able to:

1. explain why dimensionality reduction can be useful;
2. describe a principal component as a weighted combination of original features;
3. explain why numerical features should usually be standardized before PCA;
4. fit PCA using training data only;
5. interpret explained and cumulative variance;
6. visualize observations in principal-component space;
7. select components using a variance threshold; and
8. discuss the information loss and interpretability tradeoffs introduced by PCA.


## What will you build?

You will create a leakage-safe PCA workflow that compresses four related academic measures into fewer derived features. You will examine:

**Standardized features → Principal components → Explained variance → Reduced representation → Reconstruction**

> **Key principle:** PCA finds directions of maximum variance—not directions of fairness, causality, or predictive importance.


## 1. Import the libraries


In [ ]:
from io import StringIO

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 3)

print("Pandas version:", pd.__version__)


## 2. Load and prepare the dataset


In [ ]:
student_csv = """student_id,study_hours,attendance_pct,previous_score,learning_mode,programming_experience,assignments_submitted,final_score,passed
S001,5.5,92,78,In-person,Beginner,9,84,Yes
S002,3.0,75,65,Online,No prior experience,7,68,Yes
S003,1.5,61,58,Hybrid,No prior experience,5,55,No
S004,6.0,95,88,In-person,Intermediate,10,91,Yes
S005,2.0,70,62,Online,Beginner,6,63,Yes
S006,4.5,85,74,Hybrid,Beginner,8,79,Yes
S007,1.0,55,51,Online,No prior experience,4,48,No
S008,7.0,98,91,In-person,Advanced,10,95,Yes
S009,3.5,82,69,Hybrid,Beginner,8,73,Yes
S010,2.5,67,60,Online,No prior experience,6,59,No
S011,5.0,90,81,In-person,Intermediate,9,86,Yes
S012,4.0,88,76,Hybrid,Beginner,8,80,Yes
S013,2.0,,57,Online,No prior experience,5,54,No
S014,6.5,96,89,In-person,Advanced,10,93,Yes
S015,3.0,78,,Hybrid,Beginner,7,70,Yes
S016,1.5,63,55,Online,No prior experience,4,52,No
S017,5.5,91,83,In-person,Intermediate,9,88,Yes
S018,4.0,84,72,hybrid,Beginner,8,77,Yes
S019,2.5,72,64,Online,Beginner,6,65,Yes
S020,6.0,94,86,In-person,Advanced,10,90,Yes
S021,3.5,80,70,Hybrid,Beginner,7,72,Yes
S022,1.0,58,49,Online,No prior experience,3,45,No
S023,4.5,87,75,In-person,Intermediate,9,82,Yes
S024,2.0,69,59,Online,No prior experience,5,57,No
S025,5.0,89,80,Hybrid,Intermediate,9,85,Yes
S026,3.0,76,67,Online,Beginner,7,69,Yes
S027,6.5,97,90,In-person,Advanced,10,94,Yes
S028,1.5,60,53,Online,No prior experience,4,50,No
S029,4.0,83,73,Hybrid,Beginner,8,78,Yes
S030,2.5,74,63,Online,Beginner,6,64,Yes
S030,2.5,74,63,Online,Beginner,6,64,Yes
"""

students = pd.read_csv(StringIO(student_csv))
print("Dataset loaded successfully.")


In [ ]:
model_data = students.drop_duplicates().reset_index(drop=True).copy()
model_data["learning_mode"] = model_data["learning_mode"].str.strip().str.title()

numeric_features = [
    "study_hours",
    "attendance_pct",
    "previous_score",
    "assignments_submitted",
]

X = model_data[numeric_features].copy()
y = model_data["passed"].map({"No": 0, "Yes": 1})

print("Feature matrix:", X.shape)
display(X.head())


We use only continuous or ordered numerical measures. Student IDs are excluded because they have no quantitative meaning. Categorical variables are also excluded from this introductory PCA analysis because distances between one-hot indicators require careful interpretation.

`final_score` is excluded because it strongly overlaps with the outcome `passed` and would create leakage in a predictive workflow.


## 3. Why reduce dimensions?


A dataset with many related features can contain redundancy. Dimensionality reduction may:

- compress several correlated features into fewer variables;
- support two- or three-dimensional visualization;
- reduce storage and computation;
- reduce multicollinearity; and
- sometimes reduce noise.

PCA creates new, mutually perpendicular axes called **principal components**. The first component captures the greatest possible variance; each later component captures the greatest remaining variance while remaining perpendicular to earlier components.

For standardized features $z_1, z_2, \ldots, z_p$, a component has the form:

$$PC_1 = w_1z_1 + w_2z_2 + \cdots + w_pz_p$$

The weights $w_i$ are learned from the data.


## 4. Explore relationships among features


In [ ]:
correlation = X.corr()
display(correlation)

fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(correlation, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(numeric_features)), numeric_features, rotation=45, ha="right")
ax.set_yticks(range(len(numeric_features)), numeric_features)

for row in range(len(numeric_features)):
    for column in range(len(numeric_features)):
        ax.text(column, row, f"{correlation.iloc[row, column]:.2f}",
                ha="center", va="center", color="black")

ax.set_title("Correlation Among Numerical Features")
fig.colorbar(image, ax=ax, label="Correlation")
plt.tight_layout()
plt.show()


Strong correlation suggests that features share information. PCA can represent some of this shared variation with fewer components. Correlation alone does not prove that a feature should be removed.


## 5. Split before learning preprocessing


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)


Imputation, standardization, and PCA all learn parameters from data. Therefore, they must be fitted on the training set only. The test set is transformed later using those training-derived parameters.


## 6. Why standardize before PCA?


In [ ]:
scale_summary = X_train.agg(["mean", "std", "min", "max"]).T
display(scale_summary)


PCA is sensitive to measurement scale. A feature measured over a wide numerical range can dominate the variance even when it is not more important. Standardization gives each feature mean near 0 and standard deviation near 1 before PCA.


In [ ]:
preparation = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

X_train_scaled = preparation.fit_transform(X_train)
X_test_scaled = preparation.transform(X_test)

scaled_check = pd.DataFrame(X_train_scaled, columns=numeric_features).agg(["mean", "std"]).T
display(scaled_check)


Pandas reports the sample standard deviation using $n-1$, whereas `StandardScaler` uses the population definition with $n$. Therefore, the displayed Pandas standard deviations may be slightly above 1 even though scaling is correct.


## 7. Fit PCA and inspect explained variance


In [ ]:
pca_all = PCA()
X_train_pca_all = pca_all.fit_transform(X_train_scaled)
X_test_pca_all = pca_all.transform(X_test_scaled)

variance_table = pd.DataFrame({
    "component": [f"PC{i}" for i in range(1, len(numeric_features) + 1)],
    "explained_variance_ratio": pca_all.explained_variance_ratio_,
    "cumulative_variance": np.cumsum(pca_all.explained_variance_ratio_),
})

display(variance_table)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
positions = np.arange(1, len(variance_table) + 1)

ax.bar(positions, variance_table["explained_variance_ratio"],
       color="#4C78A8", edgecolor="black", label="Individual")
ax.plot(positions, variance_table["cumulative_variance"],
        color="#E45756", marker="o", linewidth=2, label="Cumulative")
ax.axhline(0.90, color="gray", linestyle="--", label="90% threshold")
ax.set_xticks(positions, variance_table["component"])
ax.set_ylim(0, 1.05)
ax.set_xlabel("Principal component")
ax.set_ylabel("Proportion of variance")
ax.set_title("Explained Variance by Principal Component")
ax.legend()
plt.tight_layout()
plt.show()


The **explained variance ratio** is the fraction of total variance captured by one component. The **cumulative variance** shows how much is retained when the first several components are kept.

A threshold such as 90% is a design choice—not a universal law. The appropriate threshold depends on the cost of information loss, downstream task, and interpretability requirements.


## 8. Interpret component weights


In [ ]:
loadings = pd.DataFrame(
    pca_all.components_.T,
    index=numeric_features,
    columns=[f"PC{i}" for i in range(1, len(numeric_features) + 1)],
)

display(loadings)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
loadings[["PC1", "PC2"]].plot(kind="bar", ax=ax, edgecolor="black")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Feature Weights for the First Two Components")
ax.set_xlabel("Original feature")
ax.set_ylabel("Component weight")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


Large absolute weights indicate features that contribute strongly to a component. The sign describes direction, but the overall sign of a principal component can be reversed without changing its meaning. Interpret patterns of relative magnitude and direction rather than treating positive as inherently good.


## 9. Visualize observations in two dimensions


In [ ]:
train_scores = pd.DataFrame(
    X_train_pca_all[:, :2], columns=["PC1", "PC2"], index=X_train.index
)
train_scores["passed"] = y_train.map({0: "No", 1: "Yes"})

fig, ax = plt.subplots(figsize=(7, 5))
colors = {"No": "#E45756", "Yes": "#4C78A8"}

for label, group in train_scores.groupby("passed"):
    ax.scatter(group["PC1"], group["PC2"], label=label,
               color=colors[label], s=75, edgecolor="black", alpha=0.8)

ax.axhline(0, color="gray", linewidth=0.7)
ax.axvline(0, color="gray", linewidth=0.7)
ax.set_xlabel("Principal component 1")
ax.set_ylabel("Principal component 2")
ax.set_title("Training Observations in PCA Space")
ax.legend(title="Passed")
plt.tight_layout()
plt.show()


The outcome labels were used only to color points after PCA was fitted. PCA itself did not use `passed`; it is an **unsupervised** transformation. Visible separation can be suggestive, but it is not a reliable performance evaluation.


## 10. Select components using a variance threshold


In [ ]:
pca_90 = PCA(n_components=0.90)
X_train_reduced = pca_90.fit_transform(X_train_scaled)
X_test_reduced = pca_90.transform(X_test_scaled)

print("Original number of features:", X_train_scaled.shape[1])
print("Components retained:", pca_90.n_components_)
print("Variance retained:", round(pca_90.explained_variance_ratio_.sum(), 3))
print("Reduced training shape:", X_train_reduced.shape)
print("Reduced test shape:", X_test_reduced.shape)


Passing a number between 0 and 1 to `n_components` asks PCA to keep the smallest number of components whose cumulative explained variance reaches that threshold.


## 11. Package the complete PCA workflow


In [ ]:
pca_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.90)),
])

train_reduced_pipeline = pca_pipeline.fit_transform(X_train)
test_reduced_pipeline = pca_pipeline.transform(X_test)

print("Pipeline output shapes:", train_reduced_pipeline.shape, test_reduced_pipeline.shape)
print("Components selected:", pca_pipeline.named_steps["pca"].n_components_)


The pipeline accepts the original DataFrame and preserves the correct order: impute, standardize, then reduce. In cross-validation, this complete pipeline must be fitted separately within each training fold.


## 12. Measure the reconstruction tradeoff


In [ ]:
reconstructed_scaled = pca_90.inverse_transform(X_train_reduced)
reconstruction_mse = np.mean((X_train_scaled - reconstructed_scaled) ** 2)

comparison = pd.DataFrame({
    "original_scaled": X_train_scaled[0],
    "reconstructed_scaled": reconstructed_scaled[0],
    "absolute_difference": np.abs(X_train_scaled[0] - reconstructed_scaled[0]),
}, index=numeric_features)

print("Mean squared reconstruction error:", round(reconstruction_mse, 4))
display(comparison)


When fewer components are retained, inverse transformation can only approximate the original standardized values. More components preserve more information but provide less compression. PCA is therefore a tradeoff, not a free improvement.


## 13. Compare thresholds


In [ ]:
threshold_results = []

for threshold in [0.70, 0.80, 0.90, 0.95, 0.99]:
    candidate = PCA(n_components=threshold)
    reduced = candidate.fit_transform(X_train_scaled)
    reconstructed = candidate.inverse_transform(reduced)
    error = np.mean((X_train_scaled - reconstructed) ** 2)
    threshold_results.append({
        "requested_threshold": threshold,
        "components": candidate.n_components_,
        "actual_variance_retained": candidate.explained_variance_ratio_.sum(),
        "reconstruction_mse": error,
    })

threshold_table = pd.DataFrame(threshold_results)
display(threshold_table)


Because components are indivisible, actual retained variance may exceed the requested threshold. Notice how reconstruction error generally decreases as more components are retained.


## 14. Guided practice


Complete these tasks:

1. Identify the component that explains the most variance.
2. Find the smallest number of components needed to retain at least 95% variance.
3. Identify the original feature with the largest absolute weight on PC1.
4. Transform `X_test` with `pca_pipeline` and confirm the number of columns.
5. Explain why calling `fit_transform(X_test)` would be incorrect.


In [ ]:
# Write your solution here.


<details>
<summary><strong>Open the suggested solution</strong></summary>

```python
# 1. Component explaining the most variance
print(variance_table.loc[variance_table["explained_variance_ratio"].idxmax(), "component"])

# 2. Components for at least 95% variance
components_95 = np.argmax(variance_table["cumulative_variance"].to_numpy() >= 0.95) + 1
print(components_95)

# 3. Largest absolute loading on PC1
print(loadings["PC1"].abs().idxmax())

# 4. Transform test data
test_reduced = pca_pipeline.transform(X_test)
print(test_reduced.shape[1])

# 5. fit_transform on X_test would learn imputation, scaling, and PCA directions from test data.
```

</details>


## 15. Challenge: Is PCA useful here?


Compare these two representations:

- all four standardized numerical features; and
- the components retained at a 90% variance threshold.

Write a recommendation on whether PCA should be used for this dataset. Consider:

1. the number of features removed;
2. the variance retained;
3. reconstruction error;
4. loss of direct feature meaning;
5. the small sample size; and
6. the intended use—visualization, compression, or prediction.

There is no requirement to use PCA merely because it is available. A defensible conclusion may be that the original features are easier to interpret and already few in number.


## 16. Common mistakes to avoid


| Mistake | Why it is a problem | Better practice |
|---|---|---|
| Applying PCA before scaling | Large-range features can dominate variance | Impute and standardize first |
| Fitting PCA before splitting | Test information influences component directions | Split first and fit on training data |
| Selecting components by eigenvalue alone | Task and information-loss costs are ignored | Inspect cumulative variance and downstream needs |
| Interpreting PCA as causal | Components describe covariance, not causes | Use cautious, descriptive language |
| Applying PCA automatically to every feature | Categorical encodings may distort distances | Justify the feature space and method |
| Assuming more compression is always better | Information and interpretability are lost | Evaluate the tradeoff explicitly |


## 17. Responsible use and limitations


- PCA preserves high-variance directions, but low-variance patterns may still matter for minority groups or rare cases.
- Components can make decisions harder to explain because each one mixes several original variables.
- A two-dimensional plot can hide information contained in later components.
- PCA does not remove bias, establish fairness, or protect privacy.
- Component patterns may shift when the population or data-collection process changes.

For high-impact decisions, examine subgroup effects, stability, interpretability, and domain meaning before adopting dimensionality reduction.


## 18. Reflection


1. What does the first principal component represent?
2. Why is standardization important before PCA?
3. What does an explained variance ratio measure?
4. Why can a component's signs be reversed without changing the PCA solution?
5. What is lost when fewer components are retained?
6. When might retaining the original features be better than applying PCA?
7. How could PCA hide a pattern important to a small subgroup?


## 19. Key takeaways


- PCA creates new orthogonal features that capture decreasing amounts of variance.
- Standardize numerical features before PCA when scales differ.
- Split data before fitting imputation, scaling, or PCA.
- Explained variance helps quantify how much information each component retains.
- Loadings connect components to their contributing original features but require cautious interpretation.
- Reducing dimensions trades information and interpretability for compression and visualization.
- PCA is an unsupervised transformation; it does not guarantee improved prediction or responsible decisions.

## Chapter 4 notebook journey complete

You have now practiced the complete pre-processing sequence:

1. **Explore and select data with Pandas**
2. **Clean real-world data**
3. **Prepare numerical and categorical features safely**
4. **Reduce numerical features with PCA**
